# Fake News Detector — with Imbalanced Data Handling

This notebook extends the original pipeline by properly handling class
imbalance between the "Fake" and "Real" news classes. Three different
strategies are used depending on what each model actually supports:

- **Multinomial Naive Bayes** — doesn't have a `class_weight` parameter,
  but `.fit()` accepts `sample_weight`. We compute balanced sample weights
  with `compute_sample_weight('balanced', y_train)` so minority-class rows
  count more during training, without touching the underlying TF-IDF matrix.
- **Logistic Regression** — supports `class_weight='balanced'` natively,
  which is the simplest and most reliable option.
- **KNN** — has no weighting mechanism at all (predictions are just a
  majority vote of neighbors), so the only way to fix imbalance is to
  resample the actual training data. We use `RandomUnderSampler` (fast,
  works well with cosine-distance KNN, avoids the "fake neighbor" noise
  that SMOTE can introduce into sparse TF-IDF space).

All resampling/weighting is fit **only on the training split**, never on
the test split, to avoid data leakage and inflated metrics.

In [ ]:
import re
import nltk
import pickle
import datetime
import pandas as pd
import numpy as np
from nltk.corpus import stopwords, wordnet
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer, PorterStemmer
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import MinMaxScaler
from sklearn.naive_bayes import MultinomialNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, classification_report
)
from sklearn.utils.class_weight import compute_sample_weight

from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.sparse import csr_matrix, hstack

## Load raw data

If running in Google Colab, mount your drive first. If running locally,
just make sure `Fake.csv` and `True.csv` are in the paths below (adjust
as needed).

In [ ]:
# Uncomment if running in Google Colab
# from google.colab import drive
# drive.mount('/content/drive')

DATA_DIR = '/content/drive/MyDrive/dataset'  # change if running locally

fake = pd.read_csv(f'{DATA_DIR}/Fake.csv')
true = pd.read_csv(f'{DATA_DIR}/True.csv')

In [ ]:
true['date'] = pd.to_datetime(true['date'], format='%B %d, %Y ', errors='coerce')
true.drop_duplicates(inplace=True)

In [ ]:
fake['date'].value_counts()

## Clean the `date` column in `fake`

In [ ]:
# Step 1: Clean basic issues
fake['date'] = fake['date'].astype(str).str.strip()
fake['date'] = fake['date'].str.replace(r'\s+', ' ', regex=True)

# Step 2: Remove NON-date rows (URLs, text)
fake = fake[~fake['date'].str.contains(r'http|www|\.com', case=False, na=False)]
fake = fake[fake['date'].str.contains(r'\d')]  # keep rows having numbers

# Step 3: Convert to datetime (AUTO handles all formats)
fake['date_parsed'] = pd.to_datetime(fake['date'], errors='coerce')

# Step 4: Check failed conversions
print("Null values:", fake['date_parsed'].isna().sum())

# Optional: remove remaining bad rows
fake = fake.dropna(subset=['date_parsed'])

# Step 5: Extract useful features (for ML)
fake['year'] = fake['date_parsed'].dt.year
fake['month'] = fake['date_parsed'].dt.month
fake['day'] = fake['date_parsed'].dt.day

# Final check
print(fake.shape)

In [ ]:
fake = fake[['title', 'text', 'subject', 'date_parsed', 'year', 'month', 'day']]

## Label and merge the two datasets

In [ ]:
fake['label'] = 0
true['label'] = 1

true['year'] = true['date'].dt.year
true['month'] = true['date'].dt.month
true['day'] = true['date'].dt.day
true.rename(columns={'date': 'date_parsed'}, inplace=True)

finel = pd.concat([fake, true])
print("Duplicates:", finel.duplicated().sum())
print(finel.isna().sum())
finel.head()

## Text preprocessing pipeline

In [ ]:
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('averaged_perceptron_tagger')
nltk.download('averaged_perceptron_tagger_eng')

STOPWORDS = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()
stemmer = PorterStemmer()

In [ ]:
def lowercase(text):
    return str(text).lower()


def remove_noise(text):
    text = re.sub(r'http\S+|www\S+|https\S+', '', text)   # URLs
    text = re.sub(r'<.*?>', '', text)                      # HTML tags
    text = re.sub(r'@\w+', '', text)                       # mentions
    text = re.sub(r'#\w+', '', text)                       # hashtags
    text = re.sub(r'\d+', '', text)                        # numbers
    text = re.sub(r'\s+', ' ', text)                        # extra whitespace
    return text.strip()


def remove_punctuation(text):
    return re.sub(r'[^\w\s]', '', text)


def tokenize(text):
    return word_tokenize(text)


def remove_stopwords(tokens):
    return [t for t in tokens if t not in STOPWORDS and len(t) > 2]


def get_wordnet_pos(tag):
    tag_map = {'J': wordnet.ADJ, 'V': wordnet.VERB,
               'N': wordnet.NOUN, 'R': wordnet.ADV}
    return tag_map.get(tag[0].upper(), wordnet.NOUN)


def lemmatize_tokens(tokens):
    pos_tags = nltk.pos_tag(tokens)   # batch tag — much faster than per-word
    return [lemmatizer.lemmatize(word, get_wordnet_pos(tag)) for word, tag in pos_tags]


def stem_tokens(tokens):
    return [stemmer.stem(t) for t in tokens]


def preprocess_pipeline(text, use_lemmatization=True):
    text = lowercase(text)
    text = remove_noise(text)
    text = remove_punctuation(text)
    tokens = tokenize(text)
    tokens = remove_stopwords(tokens)

    if use_lemmatization:
        tokens = lemmatize_tokens(tokens)
    else:
        tokens = stem_tokens(tokens)

    return ' '.join(tokens)

In [ ]:
from tqdm import tqdm
tqdm.pandas()

finel['clean_text'] = finel['text'].progress_apply(
    lambda x: preprocess_pipeline(x, use_lemmatization=True)
)
finel['clean_title'] = finel['title'].progress_apply(
    lambda x: preprocess_pipeline(x, use_lemmatization=True)
)

print(finel[['text', 'clean_text']].head())

In [ ]:
# Save checkpoint before vectorization — preprocessing is expensive,
# don't redo it if your next step crashes
finel.to_csv('preprocessed_fake_real_news.csv', index=False)
print("Saved preprocessed_fake_real_news.csv")

## Load preprocessed data & handle missing values

In [ ]:
RANDOM_STATE = 42
df = pd.read_csv(f'{DATA_DIR}/preprocessed_fake_real_news.csv')
print("Raw shape:", df.shape)
print("\nMissing values before handling:\n", df.isnull().sum())

In [ ]:
# clean_text: critical column — can't impute text, so drop rows with no text
df = df[df['clean_text'].notnull()]
df = df[df['clean_text'].str.strip() != '']

# title: not critical for this model — fill empty string if missing
df['title'] = df['title'].fillna('')

# subject: fill with 'unknown' category if missing
df['subject'] = df['subject'].fillna('unknown')

# date-related columns: drop rows missing these ONLY when building the date-feature version
df_text_only = df.copy()
df_with_date = df.dropna(subset=['year', 'month', 'day']).copy()

print("\nShape after handling missing values:")
print("  Text-only dataset:", df_text_only.shape)
print("  Text+date dataset:", df_with_date.shape)
print("  Rows dropped due to missing date:", df_text_only.shape[0] - df_with_date.shape[0])

## Check class imbalance

This is the step that was missing before — we actually look at the ratio
and visualize it, instead of assuming the dataset is balanced.

In [ ]:
print("\nClass balance (text-only df):\n", df_text_only['label'].value_counts())
print("\nClass balance (text-only df, %):\n", df_text_only['label'].value_counts(normalize=True) * 100)

counts = df_text_only['label'].value_counts()
imbalance_ratio = counts.max() / counts.min()
print(f"\nImbalance ratio (majority:minority) = {imbalance_ratio:.2f} : 1")

plt.figure(figsize=(5, 4))
sns.countplot(x='label', data=df_text_only, palette='Set2')
plt.xticks([0, 1], ['Fake (0)', 'Real (1)'])
plt.title('Class Distribution Before Balancing')
plt.ylabel('Count')
plt.tight_layout()
plt.show()

## Feature building helper (unchanged)

In [ ]:
def build_features(data, use_date=False, tfidf_params=None):
    if tfidf_params is None:
        tfidf_params = dict(max_features=50000, ngram_range=(1, 2), min_df=5, max_df=0.7)

    if use_date:
        X_text = data['clean_text']
        X_dates = data[['year', 'month', 'day']]
        y = data['label']

        X_text_train, X_text_test, X_date_train, X_date_test, y_train, y_test = train_test_split(
            X_text, X_dates, y,
            test_size=0.2, random_state=RANDOM_STATE, stratify=y
        )

        tfidf = TfidfVectorizer(**tfidf_params)
        X_text_train_vec = tfidf.fit_transform(X_text_train)
        X_text_test_vec = tfidf.transform(X_text_test)

        scaler = MinMaxScaler()
        X_date_train_scaled = scaler.fit_transform(X_date_train)
        X_date_test_scaled = scaler.transform(X_date_test)

        X_train = hstack([X_text_train_vec, csr_matrix(X_date_train_scaled)])
        X_test = hstack([X_text_test_vec, csr_matrix(X_date_test_scaled)])

        return X_train, X_test, y_train, y_test, tfidf, scaler

    else:
        X_text = data['clean_text']
        y = data['label']

        X_text_train, X_text_test, y_train, y_test = train_test_split(
            X_text, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
        )

        tfidf = TfidfVectorizer(**tfidf_params)
        X_train = tfidf.fit_transform(X_text_train)
        X_test = tfidf.transform(X_text_test)

        return X_train, X_test, y_train, y_test, tfidf, None

## Build both feature versions

In [ ]:
print("\n" + "=" * 60)
print("BUILDING FEATURE VERSIONS")
print("=" * 60)

X_train_A, X_test_A, y_train_A, y_test_A, tfidf_A, _ = build_features(df_text_only, use_date=False)
print("Version A (Text Only):     X_train", X_train_A.shape, " X_test", X_test_A.shape)

X_train_B, X_test_B, y_train_B, y_test_B, tfidf_B, scaler_B = build_features(df_with_date, use_date=True)
print("Version B (Text + Date):   X_train", X_train_B.shape, " X_test", X_test_B.shape)

## Imbalance handling

`handle_imbalance` centralizes every strategy used below:

- `'weight'` — returns `sample_weight` only (no data is changed). Used for
  Naive Bayes, which supports `sample_weight` in `.fit()`.
- `'undersample'` — randomly drops majority-class rows down to the
  minority count. Used for KNN, since it has no weighting mechanism and
  undersampling also makes KNN's expensive distance search faster.
- `'smote'` — synthetic oversampling of the minority class, available if
  you want to try it instead of undersampling (e.g. when you can't afford
  to lose majority-class rows). Not used by default here because SMOTE on
  high-dimensional sparse TF-IDF vectors tends to create less realistic
  synthetic samples than plain undersampling for this dataset size.
- `'none'` — pass the data through unchanged (used for Logistic
  Regression, which instead gets `class_weight='balanced'` directly).

In [ ]:
def handle_imbalance(X_train, y_train, method='none', random_state=RANDOM_STATE):
    """
    method: 'none' | 'weight' | 'undersample' | 'smote'
    Returns: X_train, y_train, sample_weight (sample_weight is None unless method=='weight')
    """
    if method == 'none':
        return X_train, y_train, None

    if method == 'weight':
        sample_weight = compute_sample_weight(class_weight='balanced', y=y_train)
        return X_train, y_train, sample_weight

    if method == 'undersample':
        rus = RandomUnderSampler(random_state=random_state)
        X_res, y_res = rus.fit_resample(X_train, y_train)
        return X_res, y_res, None

    if method == 'smote':
        smote = SMOTE(random_state=random_state)
        X_res, y_res = smote.fit_resample(X_train, y_train)
        return X_res, y_res, None

    raise ValueError(f"Unknown method: {method}")

## Model definitions with light tuning + imbalance handling

In [ ]:
def train_naive_bayes(X_train, y_train):
    # Naive Bayes: balance via sample_weight, not resampling
    X_bal, y_bal, sample_weight = handle_imbalance(X_train, y_train, method='weight')

    grid = GridSearchCV(MultinomialNB(), {'alpha': [0.01, 0.1, 0.5, 1.0]},
                         cv=3, scoring='f1', n_jobs=-1)
    grid.fit(X_bal, y_bal, sample_weight=sample_weight)
    print(f"   Naive Bayes best alpha: {grid.best_params_}")
    return grid.best_estimator_


def train_knn(X_train, y_train):
    # KNN: no weighting mechanism -> undersample majority class instead
    X_bal, y_bal, _ = handle_imbalance(X_train, y_train, method='undersample')
    print(f"   KNN training size after undersampling: {X_bal.shape[0]} "
          f"(was {X_train.shape[0]})")

    grid = GridSearchCV(KNeighborsClassifier(metric='cosine'), {'n_neighbors': [3, 5, 7]},
                         cv=3, scoring='f1', n_jobs=-1)
    grid.fit(X_bal, y_bal)
    print(f"   KNN best k: {grid.best_params_}")
    return grid.best_estimator_


def train_logistic_regression(X_train, y_train):
    # Logistic Regression: native class_weight support, no resampling needed
    grid = GridSearchCV(
        LogisticRegression(max_iter=1000, random_state=RANDOM_STATE, class_weight='balanced'),
        {'C': [0.1, 1, 10]}, cv=3, scoring='f1', n_jobs=-1
    )
    grid.fit(X_train, y_train)
    print(f"   Logistic Regression best C: {grid.best_params_}")
    return grid.best_estimator_


def evaluate_model(model, X_test, y_test, model_name, version_name):
    preds = model.predict(X_test)
    return {
        'Model': model_name,
        'Version': version_name,
        'Accuracy': accuracy_score(y_test, preds),
        'Precision': precision_score(y_test, preds),
        'Recall': recall_score(y_test, preds),
        'F1': f1_score(y_test, preds),
        'F1_Fake_Class': f1_score(y_test, preds, pos_label=0)
    }, preds

## Train all 3 models on both feature versions (6 total runs)

Note: the *test* sets (`X_test_A/B`, `y_test_A/B`) are always left in their
original, un-resampled, imbalanced state — that's the real-world
distribution the model will actually be evaluated against. Only the
training data is balanced.

In [ ]:
results = []
trained = {}

versions = {
    'Text Only':   (X_train_A, X_test_A, y_train_A, y_test_A),
    'Text + Date': (X_train_B, X_test_B, y_train_B, y_test_B)
}

print("\n" + "=" * 60)
print("TRAINING ALL MODELS (this takes a few minutes — KNN is slow)")
print("=" * 60)

for version_name, (X_train, X_test, y_train, y_test) in versions.items():
    print(f"\n--- {version_name} ---")

    print(" Training Naive Bayes...")
    nb = train_naive_bayes(X_train, y_train)
    m, p = evaluate_model(nb, X_test, y_test, 'Naive Bayes', version_name)
    results.append(m); trained[f'NB__{version_name}'] = (nb, p, y_test)

    print(" Training KNN...")
    knn = train_knn(X_train, y_train)
    m, p = evaluate_model(knn, X_test, y_test, 'KNN', version_name)
    results.append(m); trained[f'KNN__{version_name}'] = (knn, p, y_test)

    print(" Training Logistic Regression...")
    lr = train_logistic_regression(X_train, y_train)
    m, p = evaluate_model(lr, X_test, y_test, 'Logistic Regression', version_name)
    results.append(m); trained[f'LR__{version_name}'] = (lr, p, y_test)

## Final results table

In [ ]:
results_df = pd.DataFrame(results).sort_values('F1', ascending=False).reset_index(drop=True)

print("\n" + "=" * 70)
print("FINAL COMPARISON TABLE — ALL MODELS x BOTH VERSIONS")
print("=" * 70)
print(results_df.to_string(index=False))

results_df.to_csv(f'{DATA_DIR}/final_model_comparison.csv', index=False)

best = results_df.iloc[0]
print(f"\nBEST OVERALL: {best['Model']} ({best['Version']})")
print(f"   F1: {best['F1']:.4f}  |  Accuracy: {best['Accuracy']:.4f}  |  F1 (Fake class): {best['F1_Fake_Class']:.4f}")

## Does adding date features help?

In [ ]:
pivot = results_df.pivot(index='Model', columns='Version', values='F1')
pivot['Improvement'] = pivot['Text + Date'] - pivot['Text Only']
print(pivot)

if (pivot['Improvement'].abs() < 0.01).all():
    print("\n-> Conclusion: Date features add negligible lift (<1% F1) across all models.")
    print("  Recommendation: drop date features — text signal alone is sufficient and more generalizable.")
else:
    print("\n-> Conclusion: Date features show a meaningful difference for at least one model — inspect per-model.")

## Visualize comparison

In [ ]:
plt.figure(figsize=(11, 6))
sns.barplot(data=results_df, x='Model', y='F1', hue='Version', palette='Set2')
plt.title('Model F1-Score: Text Only vs Text + Date Features', fontsize=14)
plt.ylim(0.8, 1.0)
plt.ylabel('F1-Score')
plt.tight_layout()
plt.savefig(f'{DATA_DIR}/model_comparison_chart.png', dpi=150)
plt.show()

## Confusion matrix for the best model

In [ ]:
best_key = f"{best['Model'].replace('Naive Bayes', 'NB').replace('Logistic Regression', 'LR').replace(' ', '')}__{best['Version']}"
matches = [k for k in trained if k.split('__')[0] == best_key.split('__')[0] and k.split('__')[1] == best['Version']]

if matches:
    best_model, best_preds, best_y_test = trained[matches[0]]
    cm = confusion_matrix(best_y_test, best_preds)

    plt.figure(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['Fake', 'Real'], yticklabels=['Fake', 'Real'])
    plt.title(f"Confusion Matrix — {best['Model']} ({best['Version']})")
    plt.ylabel('Actual')
    plt.xlabel('Predicted')
    plt.tight_layout()
    plt.savefig(f'{DATA_DIR}/best_model_confusion_matrix.png', dpi=150)
    plt.show()

    print("\nClassification Report — Best Model:")
    print(classification_report(best_y_test, best_preds, target_names=['Fake', 'Real']))

print("\nAll outputs saved to the dataset folder:")
print(" - final_model_comparison.csv")
print(" - model_comparison_chart.png")
print(" - best_model_confusion_matrix.png")

## Save all trained models

In [ ]:
all_models = {
    'models': trained,
    'tfidf_A': tfidf_A,
    'tfidf_B': tfidf_B,
    'scaler_B': scaler_B
}

with open(f'{DATA_DIR}/all_models.pkl', 'wb') as f:
    pickle.dump(all_models, f)

print("All models saved successfully!")

## Reload and test on new samples

In [ ]:
with open(f'{DATA_DIR}/all_models.pkl', 'rb') as f:
    data = pickle.load(f)

trained_models = data['models']
tfidf_A = data['tfidf_A']
tfidf_B = data['tfidf_B']
scaler_B = data['scaler_B']

model, _, _ = trained_models['LR__Text Only']

true_news_samples = [
    "The government announced a new education policy aimed at improving digital literacy across rural areas.",
    "Scientists have discovered a new method to improve battery efficiency, potentially extending the life of electric vehicles.",
    "The stock market showed steady growth today as technology companies reported higher than expected earnings.",
    "A new healthcare initiative has been launched to provide affordable medical services in underserved communities.",
    "Researchers at a leading university have developed an AI system capable of early disease detection.",
    "The central bank has decided to keep interest rates unchanged following stable inflation data.",
    "An international climate summit concluded with agreements to reduce carbon emissions over the next decade.",
    "The transportation department has approved funding for new highway infrastructure projects.",
    "A major technology company released its latest smartphone model featuring improved performance and battery life.",
    "Local authorities have implemented new safety regulations to reduce road accidents in urban areas."
]

X = tfidf_A.transform(true_news_samples)
prediction = model.predict(X)
print(prediction)